## DPU Data Reuse MCP - agent test
An AI model gets the 3 MCP tools and the plain-English questions in `agent_cases.csv`. It is **not told** which tool to use.
For each question we check: (1) did it pick the right tool, (2) with the right study/PMID, (3) does its answer contain the right fact.

**How to run:** click **Run all** (about 3-5 minutes). Results are saved to `results/agent_run_<date>.csv` and one line in `results/agent_summary_log.csv`.

Uses the live app `dpu-reuse-mcp` (as Inspector Gadget) and the model service `models.ai.gpt-6-astra` (as you). Nothing is changed.

In [0]:
%pip install -q --upgrade databricks-mcp databricks-sdk nest_asyncio
%restart_python

In [0]:
APP_NAME = "dpu-reuse-mcp"
MCP_URL = "https://dpu-reuse-mcp-7474654188777746.aws.databricksapps.com/mcp"
MODEL = "models.ai.gpt-6-astra"  # model service that answers the questions
EVAL_DIR = "/Workspace/Users/ying.huang@nih.gov/Data-Product-Usability/mcp/eval"
RUN_NOTE = "run 3 of 3"

In [0]:
import json
import sys
import nest_asyncio
from databricks.sdk import WorkspaceClient
from databricks_mcp import DatabricksMCPClient

nest_asyncio.apply()
sys.path.insert(0, EVAL_DIR)
import agent_loop as al  # the small test agent (agent_loop.py in this folder)

# MCP app, called as the shared test identity Inspector Gadget
ig = json.loads(dbutils.secrets.get(catalog="models", schema="ai", key="inspector-gadget"))
host = "https://" + spark.conf.get("spark.databricks.workspaceUrl")
ig_client = WorkspaceClient(host=host, client_id=ig["client_id"], client_secret=ig["client_secret"])
mcp_client = DatabricksMCPClient(server_url=MCP_URL, workspace_client=ig_client)

# Model service, called as you
me = WorkspaceClient()

llm_tools = al.to_llm_tools(mcp_client.list_tools())
deployment_id = me.apps.get(APP_NAME).active_deployment.deployment_id


def call_tool(name, args):
    """Send one tool call to the live MCP app and return its text result."""
    r = mcp_client.call_tool(name, args)
    return r.content[0].text if r.content else ""


print("Model:", MODEL, "| live deployment:", deployment_id)
print("Tools given to the model:", [t["name"] for t in llm_tools])

In [0]:
import pandas as pd

cases = pd.read_csv(f"{EVAL_DIR}/agent_cases.csv", dtype=str, keep_default_na=False)

rows = []
for case in cases.to_dict("records"):
    try:
        res = al.run_agent(case["question"], llm_tools, call_tool, MODEL, me.config.host, me.config.authenticate)
    except Exception as e:
        res = {"answer": f"(error: {str(e)[:300]})", "calls": [], "seconds": None}
    rows.append({
        "case_id": case["case_id"], "category": case["category"], "question": case["question"],
        "expected_tools": case["expected_tools"], **al.score(case, res),
        "tool_inputs": json.dumps([c["args"] for c in res["calls"]]),
        "seconds": res["seconds"], "answer": res["answer"],
    })
    print(case["case_id"], "passed" if rows[-1]["passed"] else "FAILED")

agent_results = pd.DataFrame(rows)
display(agent_results)

In [0]:
run_time = pd.Timestamp.now(tz="America/New_York").strftime("%Y-%m-%d %H:%M")
n = len(agent_results)
agent_summary = pd.DataFrame([{
    "run_time": run_time,
    "model": MODEL,
    "deployment_id": deployment_id,
    "cases": n,
    "right_tool_%": round(100 * agent_results["tool_ok"].mean(), 1),
    "right_input_%": round(100 * agent_results["input_ok"].mean(), 1),
    "right_answer_%": round(100 * agent_results["answer_ok"].mean(), 1),
    "all_three_%": round(100 * agent_results["passed"].mean(), 1),
    "median_seconds_per_question": agent_results["seconds"].median(),
    "note": RUN_NOTE,
}])
print("How to read: right_tool = picked the expected tool(s) (or none for off-topic); right_input = passed the right phs/PMID;")
print("right_answer = final answer contains the expected fact; seconds = whole question incl. model thinking.")
display(agent_summary)
failures = agent_results[~agent_results["passed"]]
if len(failures):
    display(failures)
else:
    print("No failures: all cases passed.")

In [0]:
import os

os.makedirs(f"{EVAL_DIR}/results", exist_ok=True)
stamp = run_time.replace(" ", "_").replace(":", "")
agent_results.assign(run_time=run_time, model=MODEL, deployment_id=deployment_id).to_csv(
    f"{EVAL_DIR}/results/agent_run_{stamp}.csv", index=False)
log_path = f"{EVAL_DIR}/results/agent_summary_log.csv"
agent_summary.to_csv(log_path, mode="a", header=not os.path.exists(log_path), index=False)
print("Saved:", f"results/agent_run_{stamp}.csv", "and one line in results/agent_summary_log.csv")